In [7]:
# Disable GPU for model conversion to tflite.
# Fix for https://github.com/google-ai-edge/ai-edge-torch/issues/326
# This may be removed when the issue is resolved.
# Without this fix, the training crashes with `Unknown PJRT_DEVICE 'CUDA'`.
import os
os.environ['PJRT_DEVICE'] = 'CPU'

from dataclasses import dataclass
from typing import Optional

import torch
import torchvision
import ai_edge_torch
from ai_edge_torch.quantize.quant_config import QuantConfig
from torch.ao.quantization.quantize_pt2e import convert_pt2e, prepare_qat_pt2e
from torch.ao.quantization.quantizer.xnnpack_quantizer import XNNPACKQuantizer, get_symmetric_quantization_config
from torch.ao.quantization.quantizer import QuantizationSpec

example_inputs = (torch.randn(2, 3, 224, 224),)
model = torchvision.models.mobilenet_v2(weights='DEFAULT')
model = torch.export.export_for_training(model, example_inputs).module()

quantizer = XNNPACKQuantizer().set_global(get_symmetric_quantization_config(is_qat=True))

model = prepare_qat_pt2e(model, quantizer)
model = convert_pt2e(model)
torch.ao.quantization.move_exported_model_to_eval(model)

# edge_model = ai_edge_torch.convert(model, example_inputs, quant_config=QuantConfig(pt2e_quantizer=quantizer))

/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/ao/quantization/utils.py:408: UserWarning: must run observer before calling calculate_qparams. Returning default values.
  warnings.warn(
/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm_104) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm_105) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm_106) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/ECG/lib/python3.12/site-packages/torch/fx/graph.py:1199: UserWarning: er

GraphModule(
  (features): Module(
    (0): Module(
      (0): Module()
    )
    (1): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module()
      )
    )
    (2): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (3): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (4): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (5): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (6): Module(
      (conv): Module(
        (0): 

In [8]:
torch.save(model.state_dict(), "test.pth")

In [9]:
os.path.getsize("test.pth") / 1e6 if os.path.exists("test.pth") else 0.0

17.464779